# 07 · Merging, Joining, and Concatenating

**Goal:** learn to combine multiple DataFrames — stacking them together with `concat`, and
relational joins with `merge` (like SQL `JOIN`).

### `pd.concat` — stacking DataFrames together

Use `concat` when you have data split across multiple DataFrames with the **same structure**
(same columns, or same index) that you just want to stack.

In [1]:
import pandas as pd

jan_sales = pd.DataFrame({
    "product": ["Widget", "Gadget"],
    "revenue": [1000, 1500]
})

feb_sales = pd.DataFrame({
    "product": ["Widget", "Gadget"],
    "revenue": [1100, 1600]
})

# Stack VERTICALLY (default axis=0) -- adds more ROWS
combined = pd.concat([jan_sales, feb_sales])
print(combined)
print()
print(combined.reset_index(drop=True))    # tidy up the repeated index (0,1,0,1 -> 0,1,2,3)

  product  revenue
0  Widget     1000
1  Gadget     1500
0  Widget     1100
1  Gadget     1600

  product  revenue
0  Widget     1000
1  Gadget     1500
2  Widget     1100
3  Gadget     1600


In [2]:
# Add a "month" label so you can tell rows apart after stacking
jan_sales["month"] = "Jan"
feb_sales["month"] = "Feb"

combined = pd.concat([jan_sales, feb_sales], ignore_index=True)   # ignore_index avoids duplicate labels
print(combined)

  product  revenue month
0  Widget     1000   Jan
1  Gadget     1500   Jan
2  Widget     1100   Feb
3  Gadget     1600   Feb


In [3]:
# Stack HORIZONTALLY (axis=1) -- adds more COLUMNS, aligned by index
demographics = pd.DataFrame({"product": ["Widget", "Gadget"], "category": ["Tools", "Tech"]})
prices = pd.DataFrame({"price": [9.99, 19.99]})

side_by_side = pd.concat([demographics, prices], axis=1)
print(side_by_side)

  product category  price
0  Widget    Tools   9.99
1  Gadget     Tech  19.99


### `pd.merge` — relational joins (like SQL)

Use `merge` when you have two DataFrames related through a **key column** (like a foreign
key relationship in a database) and need to combine matching rows.

In [4]:
employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "dept_id": [10, 20, 10, 30]
})

departments = pd.DataFrame({
    "dept_id": [10, 20, 30],
    "dept_name": ["Sales", "Engineering", "Marketing"]
})

print(employees)
print()
print(departments)

   emp_id     name  dept_id
0       1    Alice       10
1       2      Bob       20
2       3  Charlie       10
3       4    Diana       30

   dept_id    dept_name
0       10        Sales
1       20  Engineering
2       30    Marketing


In [5]:
# Inner join (default): only rows where the key exists in BOTH DataFrames
merged = pd.merge(employees, departments, on="dept_id")
print(merged)

   emp_id     name  dept_id    dept_name
0       1    Alice       10        Sales
1       2      Bob       20  Engineering
2       3  Charlie       10        Sales
3       4    Diana       30    Marketing


### The four join types — `how=`

In [6]:
employees2 = pd.DataFrame({
    "emp_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "dept_id": [10, 20, 10, 99]     # 99 doesn't exist in departments!
})

departments2 = pd.DataFrame({
    "dept_id": [10, 20, 30],         # 30 has no employees
    "dept_name": ["Sales", "Engineering", "Marketing"]
})

print("INNER (only matches in both):")
print(pd.merge(employees2, departments2, on="dept_id", how="inner"))
print()

print("LEFT (all of employees2, matched dept info where possible):")
print(pd.merge(employees2, departments2, on="dept_id", how="left"))
print()

print("RIGHT (all of departments2, matched employee info where possible):")
print(pd.merge(employees2, departments2, on="dept_id", how="right"))
print()

print("OUTER (everything from both, matched where possible):")
print(pd.merge(employees2, departments2, on="dept_id", how="outer"))

INNER (only matches in both):
   emp_id     name  dept_id    dept_name
0       1    Alice       10        Sales
1       2      Bob       20  Engineering
2       3  Charlie       10        Sales

LEFT (all of employees2, matched dept info where possible):
   emp_id     name  dept_id    dept_name
0       1    Alice       10        Sales
1       2      Bob       20  Engineering
2       3  Charlie       10        Sales
3       4    Diana       99          NaN

RIGHT (all of departments2, matched employee info where possible):
   emp_id     name  dept_id    dept_name
0     1.0    Alice       10        Sales
1     3.0  Charlie       10        Sales
2     2.0      Bob       20  Engineering
3     NaN      NaN       30    Marketing

OUTER (everything from both, matched where possible):
   emp_id     name  dept_id    dept_name
0     1.0    Alice       10        Sales
1     3.0  Charlie       10        Sales
2     2.0      Bob       20  Engineering
3     NaN      NaN       30    Marketing
4     4

### Join type cheat-sheet

| `how=` | Keeps |
|---|---|
| `"inner"` (default) | Only rows with a matching key in BOTH DataFrames |
| `"left"` | All rows from the LEFT DataFrame, matched data where available (NaN otherwise) |
| `"right"` | All rows from the RIGHT DataFrame, matched data where available (NaN otherwise) |
| `"outer"` | Every row from BOTH DataFrames, matched where possible (NaN where not) |

### Merging on differently-named columns

In [7]:
orders = pd.DataFrame({
    "order_id": [1, 2, 3],
    "customer_id": [101, 102, 101]
})

customers = pd.DataFrame({
    "id": [101, 102, 103],
    "customer_name": ["Alice", "Bob", "Charlie"]
})

merged = pd.merge(orders, customers, left_on="customer_id", right_on="id")
print(merged)

   order_id  customer_id   id customer_name
0         1          101  101         Alice
1         2          102  102           Bob
2         3          101  101         Alice


### Merging on the index instead of a column

In [8]:
left = pd.DataFrame({"value_a": [1, 2, 3]}, index=["x", "y", "z"])
right = pd.DataFrame({"value_b": [10, 20, 30]}, index=["x", "y", "w"])

# left_index / right_index tell merge to use the INDEX as the join key
merged = pd.merge(left, right, left_index=True, right_index=True, how="outer")
print(merged)

# .join() is a convenient shorthand specifically for index-based merges
print(left.join(right, how="outer"))

   value_a  value_b
w      NaN     30.0
x      1.0     10.0
y      2.0     20.0
z      3.0      NaN
   value_a  value_b
w      NaN     30.0
x      1.0     10.0
y      2.0     20.0
z      3.0      NaN


### Handling overlapping column names

If both DataFrames have a column with the same name (that isn't the join key), pandas
automatically adds suffixes to disambiguate — customizable via `suffixes=`.

In [9]:
df_2023 = pd.DataFrame({"product": ["Widget", "Gadget"], "price": [10, 20]})
df_2024 = pd.DataFrame({"product": ["Widget", "Gadget"], "price": [12, 22]})

merged = pd.merge(df_2023, df_2024, on="product", suffixes=("_2023", "_2024"))
print(merged)

  product  price_2023  price_2024
0  Widget          10          12
1  Gadget          20          22


### 🧠 Quick check

1. When should you use `pd.concat` instead of `pd.merge`?
2. What's the difference between a `"left"` join and an `"inner"` join?
3. What happens to unmatched rows in an `"outer"` join?

<details>
<summary>Answers</summary>

1. When you have data with the same structure (same columns or same index) that you simply
   want to stack together, rather than combine based on a relational key.
2. An inner join keeps ONLY rows with a match in both DataFrames; a left join keeps ALL rows
   from the left DataFrame, filling in `NaN` for any unmatched right-side columns.
3. They're kept (nothing is dropped), with `NaN` filled in for whichever side didn't have a
   matching key.
</details>

### ✍️ Practice

1. Create two DataFrames of quarterly sales (`Q1`, `Q2`) with the same columns, and combine
   them into one DataFrame with `pd.concat`, adding a `quarter` label column first.
2. Create an `orders` DataFrame and a `products` DataFrame linked by `product_id`, then merge
   them with an inner join.
3. Redo the same merge as a left join and observe what happens to orders referencing a
   product_id that doesn't exist in `products`.
4. Merge two DataFrames that share a column name other than the join key, and use `suffixes`
   to distinguish the two versions.

Continue to **`08_datetime_handling.ipynb`** next.